# I-MR chart and signal-rule walkthrough

This is an exploratory chart of one anonymous SECOM measurement chosen by a transparent rule: first feature in file order with no missing values and more than one distinct value. The observation axis is sequence number. The chart is not a production alarm system and does not identify a physical process parameter or cause.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.secom_data import audit_secom, load_secom
from src.spc import compute_imr_limits, western_electric_signals
from src.plotting import plot_imr

## Choose and inspect a candidate measurement

Do not use pass/fail labels to choose this variable. Verify that the file order is monotonic in timestamp and that the selected feature has no missing readings, so moving ranges compare adjacent recorded observations.

In [ ]:
dataset = load_secom(ROOT / 'data' / 'raw')
audit = audit_secom(dataset)
assert dataset.timestamps.is_monotonic_increasing
profile = audit['feature_profile']
candidates = profile.index[(profile['missing_count'] == 0) & (profile['distinct_non_missing'] > 1)]
feature_name = candidates[0]
values = dataset.features[feature_name].astype(float).tolist()
feature_name, len(values), profile.loc[feature_name].to_dict()

## Calculate limits and signals

The I chart estimates the process level; the MR chart estimates adjacent step-to-step variation. Rule flags are statistical investigation prompts, not proof of defects. The rule boundaries in this implementation are strict: equality with a zone boundary is not counted as beyond it.

In [ ]:
limits = compute_imr_limits(values)
signals = western_electric_signals(values, limits.individuals_center, limits.sigma_estimate)
print(limits)
print('Signal count by rule:', {rule: sum(signal.rule == rule for signal in signals) for rule in range(1, 5)})
figure = plot_imr(values, limits, title=f'Exploratory I-MR: {feature_name} (anonymous feature)', signals=signals)
figure

## Interpretation prompts

1. Does the MR chart show any unusually large adjacent change?
2. Which observations were flagged, and by which rule?
3. What evidence would an engineer need before linking a signal to a tool, recipe, or lithography/etch/deposition cause?
4. Why is this chart not enough to claim the process is stable or capable?